# Music Genre Classification Using Audio Signal Features and Machine Learning

**Authors:** Kristine Vergara, Richard Oh

---

## Project Overview

This notebook demonstrates how to build a machine learning model that automatically classifies songs into musical genres based on audio features extracted from raw audio files.

### Dataset: GTZAN
- 1,000 audio clips (30 seconds each)
- 10 genres: blues, classical, country, disco, hiphop, jazz, metal, pop, reggae, rock
- 100 tracks per genre

### Features Extracted:
1. **Mel-Frequency Cepstral Coefficients (MFCCs)** - Capture timbral texture
2. **Chroma features** - Represent pitch class distribution
3. **Spectral centroid** - Brightness of sound
4. **Spectral roll-off** - Shape of spectral energy
5. **Zero crossing rate** - Rate of sign changes
6. **Tempo** - Beats per minute

### Models Trained:
- Support Vector Machine (SVM)
- Random Forest
- K-Nearest Neighbors (KNN)
- Feedforward Neural Network

---

## 1. Setup and Imports

In [ ]:
# Standard libraries
import os
import warnings
warnings.filterwarnings('ignore')

# Data manipulation
import numpy as np
import pandas as pd

# Audio processing
import librosa
import librosa.display

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, f1_score, precision_score, 
                              recall_score, classification_report, 
                              confusion_matrix)

# Neural Network
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Utilities
from tqdm import tqdm
import joblib

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("All libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")
print(f"Librosa version: {librosa.__version__}")

## 2. Configuration

In [ ]:
# Configuration
DATA_DIR = './gtzan'  # Path to GTZAN dataset
FEATURES_FILE = 'features.csv'
PROCESSED_DIR = './data'
MODELS_DIR = './models'
RESULTS_DIR = './results'

# Genre labels
GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop',
          'jazz', 'metal', 'pop', 'reggae', 'rock']

# Feature extraction parameters
N_MFCC = 40  # Number of MFCC coefficients
DURATION = 30  # Audio duration in seconds

# Training parameters
TEST_SIZE = 0.2
RANDOM_STATE = 42

# Create directories
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Configuration set!")
print(f"Genres: {GENRES}")

## 3. Feature Extraction

### Feature Details:
- **MFCCs (40 coefficients)**: Mean and standard deviation → 80 features
- **Chroma (12 bins)**: Mean and standard deviation → 24 features
- **Spectral Centroid**: Mean and standard deviation → 2 features
- **Spectral Roll-off**: Mean and standard deviation → 2 features
- **Zero Crossing Rate**: Mean and standard deviation → 2 features
- **Tempo**: 1 feature

**Total: 111 features**

In [ ]:
def extract_features(file_path, n_mfcc=40, duration=30):
    """
    Extract audio features from a single audio file.
    
    Parameters:
    -----------
    file_path : str
        Path to the audio file
    n_mfcc : int
        Number of MFCC coefficients to extract
    duration : float
        Duration of audio to load (in seconds)
    
    Returns:
    --------
    np.ndarray or None
        Feature vector of shape (111,) or None if extraction fails
    """
    try:
        # Load audio file
        y, sr = librosa.load(file_path, duration=duration, mono=True)
        
        # 1. MFCCs - Capture timbral characteristics
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
        mfcc_mean = np.mean(mfccs, axis=1)
        mfcc_std = np.std(mfccs, axis=1)
        
        # 2. Chroma features - Pitch class distribution
        chroma = librosa.feature.chroma_stft(y=y, sr=sr)
        chroma_mean = np.mean(chroma, axis=1)
        chroma_std = np.std(chroma, axis=1)
        
        # 3. Spectral Centroid - Brightness measure
        centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
        centroid_mean = np.mean(centroid)
        centroid_std = np.std(centroid)
        
        # 4. Spectral Roll-off - Spectral shape measure
        rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
        rolloff_mean = np.mean(rolloff)
        rolloff_std = np.std(rolloff)
        
        # 5. Zero Crossing Rate - Signal changes
        zcr = librosa.feature.zero_crossing_rate(y)
        zcr_mean = np.mean(zcr)
        zcr_std = np.std(zcr)
        
        # 6. Tempo - Beat information
        tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
        
        # Concatenate all features
        features = np.concatenate([
            mfcc_mean, mfcc_std,           # 80 features
            chroma_mean, chroma_std,       # 24 features
            [centroid_mean, centroid_std], # 2 features
            [rolloff_mean, rolloff_std],   # 2 features
            [zcr_mean, zcr_std],           # 2 features
            [float(tempo)]                 # 1 feature
        ])
        
        return features
        
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None


def build_dataset(data_dir, n_mfcc=40, duration=30):
    """
    Build a feature dataset from the GTZAN directory structure.
    
    Parameters:
    -----------
    data_dir : str
        Root directory containing genre subfolders
    n_mfcc : int
        Number of MFCC coefficients
    duration : float
        Audio duration to load
    
    Returns:
    --------
    pd.DataFrame
        DataFrame with features and labels
    """
    rows = []
    
    for genre in GENRES:
        genre_path = os.path.join(data_dir, genre)
        
        if not os.path.isdir(genre_path):
            print(f"Warning: Directory not found: {genre_path}")
            continue
        
        files = [f for f in os.listdir(genre_path) if f.endswith('.wav')]
        print(f"Processing {genre} ({len(files)} files)...")
        
        for fname in tqdm(files, desc=f"  {genre}", leave=False):
            fpath = os.path.join(genre_path, fname)
            feats = extract_features(fpath, n_mfcc=n_mfcc, duration=duration)
            
            if feats is not None:
                rows.append({
                    **{f"feat_{i}": v for i, v in enumerate(feats)},
                    "label": genre,
                    "filename": fname
                })
    
    return pd.DataFrame(rows)


print("Feature extraction functions defined!")

In [ ]:
# Check if features already exist
if os.path.exists(FEATURES_FILE):
    print(f"Loading existing features from {FEATURES_FILE}...")
    df = pd.read_csv(FEATURES_FILE)
else:
    print("Extracting features from audio files...")
    print("This may take a while...")
    df = build_dataset(DATA_DIR, n_mfcc=N_MFCC, duration=DURATION)
    df.to_csv(FEATURES_FILE, index=False)
    print(f"Features saved to {FEATURES_FILE}")

print(f"\nDataset shape: {df.shape}")
print(f"\nSamples per genre:")
print(df['label'].value_counts())

## 4. Data Preprocessing

### Steps:
1. Separate features and labels
2. Encode labels to integers
3. Split into train/test sets (80/20)
4. Normalize features (StandardScaler)

In [ ]:
# Prepare features and labels
feature_cols = [c for c in df.columns if c.startswith('feat_')]
X = df[feature_cols].values
y_raw = df['label'].values

print(f"Feature matrix shape: {X.shape}")
print(f"Number of samples: {len(y_raw)}")
print(f"Number of features: {X.shape[1]}")

In [ ]:
# Encode labels
le = LabelEncoder()
y = le.fit_transform(y_raw)

print("Label encoding:")
for i, genre in enumerate(le.classes_):
    print(f"  {genre}: {i}")

In [ ]:
# Train/Test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=TEST_SIZE, 
    random_state=RANDOM_STATE, 
    stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

In [ ]:
# Normalize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features normalized!")
print(f"Training feature mean: {X_train_scaled.mean():.6f}")
print(f"Training feature std: {X_train_scaled.std():.6f}")

In [ ]:
# Save preprocessed data
np.save(os.path.join(PROCESSED_DIR, 'X_train.npy'), X_train_scaled)
np.save(os.path.join(PROCESSED_DIR, 'X_test.npy'), X_test_scaled)
np.save(os.path.join(PROCESSED_DIR, 'y_train.npy'), y_train)
np.save(os.path.join(PROCESSED_DIR, 'y_test.npy'), y_test)
joblib.dump(scaler, os.path.join(PROCESSED_DIR, 'scaler.pkl'))
joblib.dump(le, os.path.join(PROCESSED_DIR, 'label_encoder.pkl'))

print(f"Preprocessed data saved to {PROCESSED_DIR}/")

## 5. Model Training

We train and compare four classifiers:
1. **Support Vector Machine (SVM)** - Good for high-dimensional data
2. **Random Forest** - Robust ensemble method
3. **K-Nearest Neighbors (KNN)** - Simple distance-based classifier
4. **Neural Network** - Deep learning approach

In [ ]:
# Dictionary to store results
results = {}
models = {}

### 5.1 Support Vector Machine (SVM)

In [ ]:
print("Training SVM (RBF kernel)...")

svm = SVC(
    kernel='rbf',
    C=10,
    gamma='scale',
    probability=True,
    random_state=RANDOM_STATE
)
svm.fit(X_train_scaled, y_train)

# Predictions
svm_pred = svm.predict(X_test_scaled)
svm_acc = accuracy_score(y_test, svm_pred)
svm_f1 = f1_score(y_test, svm_pred, average='weighted')

results['SVM'] = {'accuracy': svm_acc, 'f1': svm_f1}
models['SVM'] = svm

# Save model
joblib.dump(svm, os.path.join(MODELS_DIR, 'svm.pkl'))

print(f"SVM Accuracy: {svm_acc:.4f}")
print(f"SVM F1-Score: {svm_f1:.4f}")

### 5.2 Random Forest

In [ ]:
print("Training Random Forest...")

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf.fit(X_train_scaled, y_train)

# Predictions
rf_pred = rf.predict(X_test_scaled)
rf_acc = accuracy_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred, average='weighted')

results['Random Forest'] = {'accuracy': rf_acc, 'f1': rf_f1}
models['Random Forest'] = rf

# Save model
joblib.dump(rf, os.path.join(MODELS_DIR, 'random_forest.pkl'))

print(f"Random Forest Accuracy: {rf_acc:.4f}")
print(f"Random Forest F1-Score: {rf_f1:.4f}")

### 5.3 K-Nearest Neighbors (KNN)

In [ ]:
print("Training KNN (k=5)...")

knn = KNeighborsClassifier(
    n_neighbors=5,
    metric='euclidean',
    n_jobs=-1
)
knn.fit(X_train_scaled, y_train)

# Predictions
knn_pred = knn.predict(X_test_scaled)
knn_acc = accuracy_score(y_test, knn_pred)
knn_f1 = f1_score(y_test, knn_pred, average='weighted')

results['KNN'] = {'accuracy': knn_acc, 'f1': knn_f1}
models['KNN'] = knn

# Save model
joblib.dump(knn, os.path.join(MODELS_DIR, 'knn.pkl'))

print(f"KNN Accuracy: {knn_acc:.4f}")
print(f"KNN F1-Score: {knn_f1:.4f}")

### 5.4 Neural Network

In [ ]:
def build_neural_network(input_dim, num_classes):
    """
    Build a feedforward neural network for genre classification.
    """
    model = Sequential([
        # First hidden layer
        Dense(512, activation='relu', input_shape=(input_dim,)),
        BatchNormalization(),
        Dropout(0.4),
        
        # Second hidden layer
        Dense(256, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        
        # Third hidden layer
        Dense(128, activation='relu'),
        Dropout(0.2),
        
        # Output layer
        Dense(num_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

print("Neural network architecture defined!")

In [ ]:
print("Training Neural Network...")

num_classes = len(np.unique(y_train))
nn = build_neural_network(X_train_scaled.shape[1], num_classes)

nn.summary()

In [ ]:
# Callbacks for training
callbacks = [
    EarlyStopping(
        patience=15, 
        restore_best_weights=True, 
        monitor='val_accuracy',
        verbose=1
    ),
    ReduceLROnPlateau(
        factor=0.5, 
        patience=7, 
        min_lr=1e-5, 
        monitor='val_loss',
        verbose=1
    )
]

# Train the model
history = nn.fit(
    X_train_scaled, y_train,
    epochs=200,
    batch_size=32,
    validation_data=(X_test_scaled, y_test),
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# Neural Network predictions
nn_pred = np.argmax(nn.predict(X_test_scaled), axis=1)
nn_acc = accuracy_score(y_test, nn_pred)
nn_f1 = f1_score(y_test, nn_pred, average='weighted')

results['Neural Net'] = {'accuracy': nn_acc, 'f1': nn_f1}
models['Neural Net'] = nn

# Save model
nn.save(os.path.join(MODELS_DIR, 'neural_net.keras'))

print(f"\nNeural Net Accuracy: {nn_acc:.4f}")
print(f"Neural Net F1-Score: {nn_f1:.4f}")

## 6. Model Evaluation

### 6.1 Performance Summary

In [ ]:
# Create results DataFrame
results_df = pd.DataFrame(results).T
results_df = results_df.round(4)
results_df = results_df.sort_values('accuracy', ascending=False)

print("="*50)
print("         MODEL PERFORMANCE SUMMARY")
print("="*50)
print(results_df.to_string())
print("="*50)

### 6.2 Model Comparison Bar Chart

In [ ]:
# Bar chart comparing model performance
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(results_df.index))
width = 0.35

bars1 = ax.bar(x - width/2, results_df['accuracy'], width, 
               label='Accuracy', color='steelblue', alpha=0.85)
bars2 = ax.bar(x + width/2, results_df['f1'], width, 
               label='F1 Score', color='darkorange', alpha=0.85)

ax.set_ylim(0, 1.05)
ax.set_xticks(x)
ax.set_xticklabels(results_df.index, fontsize=11)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance Comparison', fontsize=14, pad=14)
ax.legend(fontsize=11)
ax.grid(axis='y', linestyle='--', alpha=0.4)

# Add value labels
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"Saved: {RESULTS_DIR}/model_comparison.png")

### 6.3 Confusion Matrices

In [ ]:
def plot_confusion_matrix(y_true, y_pred, model_name, save_path=None):
    """
    Plot a confusion matrix heatmap.
    """
    cm = confusion_matrix(y_true, y_pred)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=GENRES,
        yticklabels=GENRES,
        ax=ax,
        linewidths=0.5
    )
    ax.set_title(f'Confusion Matrix — {model_name}', fontsize=14, pad=16)
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('Actual', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    
    plt.show()

# Plot confusion matrices for each model
predictions = {
    'SVM': svm_pred,
    'Random Forest': rf_pred,
    'KNN': knn_pred,
    'Neural Net': nn_pred
}

for name, pred in predictions.items():
    print(f"\n{name} Confusion Matrix:")
    plot_confusion_matrix(
        y_test, pred, name,
        save_path=os.path.join(RESULTS_DIR, f'confusion_{name.lower().replace(" ", "_")}.png')
    )

### 6.4 Classification Reports

In [ ]:
# Detailed classification reports
for name, pred in predictions.items():
    print(f"\n{'='*60}")
    print(f"Classification Report - {name}")
    print('='*60)
    print(classification_report(y_test, pred, target_names=GENRES))

### 6.5 Neural Network Training Curves

In [ ]:
# Plot neural network training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Accuracy
axes[0].plot(history.history['accuracy'], label='Train accuracy')
axes[0].plot(history.history['val_accuracy'], label='Val accuracy')
axes[0].set_title('Neural Net — Accuracy', fontsize=12)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(linestyle='--', alpha=0.4)

# Loss
axes[1].plot(history.history['loss'], label='Train loss')
axes[1].plot(history.history['val_loss'], label='Val loss')
axes[1].set_title('Neural Net — Loss', fontsize=12)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'nn_training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"Saved: {RESULTS_DIR}/nn_training_curves.png")

### 6.6 Per-Genre Performance Analysis

In [ ]:
# Calculate per-genre accuracy for the best model
best_model_name = max(results, key=lambda x: results[x]['accuracy'])
best_pred = predictions[best_model_name]

print(f"Best model: {best_model_name}")
print("\nPer-Genre Accuracy:")

genre_accuracy = {}
for i, genre in enumerate(GENRES):
    mask = y_test == i
    if mask.sum() > 0:
        genre_acc = (best_pred[mask] == i).mean()
        genre_accuracy[genre] = genre_acc
        print(f"  {genre:<12}: {genre_acc:.4f}")

In [ ]:
# Visualize per-genre accuracy
fig, ax = plt.subplots(figsize=(10, 6))

genres = list(genre_accuracy.keys())
accuracies = list(genre_accuracy.values())

colors = plt.cm.viridis(np.linspace(0, 1, len(genres)))
bars = ax.barh(genres, accuracies, color=colors)

ax.set_xlim(0, 1.05)
ax.set_xlabel('Accuracy', fontsize=12)
ax.set_title(f'Per-Genre Accuracy ({best_model_name})', fontsize=14)
ax.axvline(x=results[best_model_name]['accuracy'], color='red', 
           linestyle='--', label=f"Overall: {results[best_model_name]['accuracy']:.3f}")
ax.legend()

# Add value labels
for bar, acc in zip(bars, accuracies):
    ax.text(acc + 0.01, bar.get_y() + bar.get_height()/2,
            f'{acc:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'per_genre_accuracy.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7. Feature Importance Analysis (Random Forest)

In [ ]:
# Get feature importance from Random Forest
feature_names = [f'feat_{i}' for i in range(X_train_scaled.shape[1])]
importances = rf.feature_importances_

# Create feature importance DataFrame
feat_imp_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

# Plot top 20 features
fig, ax = plt.subplots(figsize=(10, 8))

top_features = feat_imp_df.head(20)
colors = plt.cm.Blues(np.linspace(0.4, 1, len(top_features)))

ax.barh(range(len(top_features)), top_features['importance'], color=colors)
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['feature'])
ax.invert_yaxis()
ax.set_xlabel('Importance', fontsize=12)
ax.set_title('Top 20 Feature Importances (Random Forest)', fontsize=14)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

## 8. Results Summary and Discussion

In [ ]:
print("="*60)
print("           FINAL RESULTS SUMMARY")
print("="*60)
print()
print("Model Performance:")
print("-"*40)
for name, scores in sorted(results.items(), key=lambda x: -x[1]['accuracy']):
    print(f"  {name:<20} Accuracy: {scores['accuracy']:.4f}  F1: {scores['f1']:.4f}")
print()
print(f"Best Model: {best_model_name}")
print(f"Best Accuracy: {results[best_model_name]['accuracy']:.4f}")
print()
print("Generated Visualizations:")
print("-"*40)
print("  - model_comparison.png")
print("  - confusion_*.png (4 files)")
print("  - nn_training_curves.png")
print("  - per_genre_accuracy.png")
print("  - feature_importance.png")
print()
print("Saved Models:")
print("-"*40)
print("  - svm.pkl")
print("  - random_forest.pkl")
print("  - knn.pkl")
print("  - neural_net.keras")
print("="*60)

## 9. Discussion and Limitations

### Key Findings:
1. The **SVM with RBF kernel** typically performs well on this task due to its ability to handle high-dimensional feature spaces.
2. **Random Forest** provides interpretable feature importance, showing which audio features are most discriminative.
3. **Classical genres** (classical, jazz) tend to be easier to classify due to distinctive spectral characteristics.
4. **Rock and country** may be confused due to similar instrumentation.

### Limitations:
1. **Dataset size**: GTZAN has only 100 samples per genre, limiting model generalization.
2. **Feature limitations**: Hand-crafted features may not capture all genre-relevant information.
3. **Temporal information**: Mean/std aggregation loses temporal structure.

### Potential Improvements:
1. **CNNs on Spectrograms**: Use convolutional neural networks directly on mel-spectrograms to preserve temporal information.
2. **Data Augmentation**: Apply pitch shifting, time stretching, and noise injection to increase training data.
3. **Larger Datasets**: Use MagnaTagATune or Million Song Dataset for better generalization.
4. **Ensemble Methods**: Combine predictions from multiple models for improved accuracy.

---

## 10. References

1. Tzanetakis, G., & Cook, P. (2002). Musical genre classification of audio signals. IEEE Transactions on Speech and Audio Processing, 10(5), 293-302.

2. Librosa: McFee, B., et al. (2015). librosa: Audio and Music Signal Analysis in Python. Proceedings of the 14th Python in Science Conference.

3. GTZAN Dataset: https://www.kaggle.com/datasets/andradaolteanu/gtzan-dataset-music-genre-classification

---